In [38]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import re
import ast

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"
tests = {}

## Helper functions

In [39]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Generate test dataset

In [47]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "type": "Python" or "JSON" or "Regex"
                "solution_criteria": "key criteria for evaluating the solution"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
        """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "````json")
    text = chat(messages, stop_sequences=["````"])
    return json.loads(text)

In [48]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## Set testing workflow

In [42]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    

def grade_syntax(task, response):
    if "Python" in task:
        return validate_python(response)
    elif "JSON" in task:
        return validate_json(response)
    elif "Regex" in task:
        return validate_regex(response)
    else:
        raise ValueError("Unknown task type")

In [49]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

Respond with only the code, JSON, or regex that solves the task. Do not include any explanations or comments.

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

#########################################################################

def grade_by_model(test_case, output):
    # Create evaluation prompt

    task = test_case["task"]
    solution = output

    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task:
    <task>
    {task}
    </task>

    Solution: 
    <solution>
    {solution}
    </solution>

    Solution Criteria: 
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

#########################################################################

def run_test_case(test_case):
    """Runs a test case and evaluates the output"""
    output = run_prompt(test_case)
    
    # TODO: Grading
    model_eval = grade_by_model(test_case, output)
    model_score = model_eval.get("score")
    model_reasoning = model_eval.get("reasoning")

    # TODO: Syntax grading
    syntax_score = grade_syntax(test_case["type"], output)

    # Combine scores (e.g., weighted average)
    score = 0 if syntax_score == 0 else model_score
    pass_syntax = syntax_score > 0

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": model_reasoning,
        "pass_syntax": pass_syntax
    }

#########################################################################

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = sum(r["score"] for r in results) / len(results)
    print(f"Average Score: {average_score}")
    final_results = {
        "average_score": average_score,
        "individual_results": results
    }
    
    return final_results

## Name and store results

In [50]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

test_name = input("Enter a name for this test run: ")

results = run_eval(dataset)
tests[test_name] = results


Average Score: 7.666666666666667


In [51]:
print(json.dumps(results, indent=2))

{
  "average_score": 7.666666666666667,
  "individual_results": [
    {
      "output": "\nimport re\n\ndef parse_s3_uri(uri):\n    pattern = r's3://([a-zA-Z0-9._-]+)/(.+)'\n    match = re.match(pattern, uri)\n    if match:\n        return {\n            'bucket': match.group(1),\n            'key': match.group(2)\n        }\n    return None\n\n# Example usage\ns3_uri = 's3://my-bucket/path/to/object'\nresult = parse_s3_uri(s3_uri)\nprint(result)\n",
      "test_case": {
        "task": "Parse an AWS S3 bucket name and key from an S3 URI in the format 's3://bucket-name/path/to/object' and extract both components",
        "type": "Regex",
        "solution_criteria": "The regex should correctly capture the bucket name and object key separately, handling paths with multiple slashes and special characters (hyphens, dots, underscores)"
      },
      "score": 7,
      "reasoning": "The solution meets the basic requirements and correctly handles the specified use case with proper regex gro

In [52]:
for test in tests:
    print(f"Test: {test}")
    print(f"Average Score: {tests[test]['average_score']}")

Test: test #1
Average Score: 6.666666666666667
Test: test #2
Average Score: 7.666666666666667
